# 502: Decline Statistics by Scenario Type

This notebook calculates decline statistics (delta_warming = peak - 2100) for three scenario groups:
1. NZCO2 scenarios (zeroed after net-zero CO2)
2. NZKyoto scenarios (zeroed after net-zero Kyoto GHG)
3. Base scenarios (no NZCO2 or NZKyoto suffix)

In [1]:
import os
from pathlib import Path

import dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

dotenv.load_dotenv()

True

In [2]:
# Configuration
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
PLOT_FOLDER = OUTPUT_FOLDER / "plots"
PLOT_FOLDER.mkdir(parents=True, exist_ok=True)

# Input metrics file
MANIFEST_ID = "regenerated"
METRICS_FILE = OUTPUT_FOLDER / f"500b_component_metrics_extended_{MANIFEST_ID}.csv"

In [3]:
OUTPUT_FOLDER

PosixPath('/Users/hoegner/GitHub/ar7_netzero_assessment/analysis/output_revision')

In [4]:
# Load metrics
metrics_df = pd.read_csv(METRICS_FILE, index_col=0)
print(f"Total rows: {len(metrics_df)}")

Total rows: 1405200


In [5]:
metrics_df.columns

Index(['model', 'scenario', 'run_id', 'peak_year', 'year_netzero_co2',
       'GSAT|ANTHROPOGENIC|peak', 'GSAT|ANTHROPOGENIC|netzero_co2',
       'GSAT|ANTHROPOGENIC|2100', 'GSAT|NonCO2_GHG|peak_gsat',
       'GSAT|NonCO2_GHG|2100', 'GSAT|NonCO2_GHG|delta_peak_gsat',
       'GSAT|NonCO2_GHG|netzero_co2', 'GSAT|NonCO2_GHG|delta_netzero_co2',
       'GSAT|Residual|peak_gsat', 'GSAT|Residual|2100',
       'GSAT|Residual|delta_peak_gsat', 'GSAT|Residual|netzero_co2',
       'GSAT|Residual|delta_netzero_co2', 'GSAT|CO2_NZCO2|peak_gsat',
       'GSAT|CO2_NZCO2|2100', 'GSAT|CO2_NZCO2|delta_peak_gsat',
       'GSAT|CO2_NZCO2|netzero_co2', 'GSAT|CO2_NZCO2|delta_netzero_co2',
       'GSAT|CO2_Negative|peak_gsat', 'GSAT|CO2_Negative|2100',
       'GSAT|CO2_Negative|delta_peak_gsat', 'GSAT|CO2_Negative|netzero_co2',
       'GSAT|CO2_Negative|delta_netzero_co2', 'scenario_group',
       'GSAT|ANTHROPOGENIC|delta_netzero_co2',
       'GSAT|ANTHROPOGENIC|delta_peak_gsat', 'is_decline'],
      dtype='

In [6]:
def calculate_scenario_stats(df, warming_column):
    """
    Calculate per-scenario statistics for delta_warming.
    
    Returns DataFrame with median, 34-67 percentile, and 5-95 percentile per scenario.
    """
    stats = df.groupby(["model", "scenario", "scenario_group"]).agg(
        n_ensemble=pd.NamedAgg(column="run_id", aggfunc="count"),
        median=pd.NamedAgg(column=warming_column, aggfunc="median"),
        p5=pd.NamedAgg(column=warming_column, aggfunc=lambda x: np.percentile(x, 5)),
        p10=pd.NamedAgg(column=warming_column, aggfunc=lambda x: np.percentile(x, 10)),
        p17=pd.NamedAgg(column=warming_column, aggfunc=lambda x: np.percentile(x, 17)),
        p34=pd.NamedAgg(column=warming_column, aggfunc=lambda x: np.percentile(x, 34)),
        p67=pd.NamedAgg(column=warming_column, aggfunc=lambda x: np.percentile(x, 67)),
        p83=pd.NamedAgg(column=warming_column, aggfunc=lambda x: np.percentile(x, 83)),
        p90=pd.NamedAgg(column=warming_column, aggfunc=lambda x: np.percentile(x, 90)),
        p95=pd.NamedAgg(column=warming_column, aggfunc=lambda x: np.percentile(x, 95)),
    ).reset_index()
    return stats


def calculate_group_summary(scenario_stats):
    """
    Calculate summary statistics across all scenarios in each scenario_group.

    Returns DataFrame with:
    - n_scenarios
    - median of medians
    - quantiles of scenario-level quantiles
    """

    group_summary = scenario_stats.groupby("scenario_group").apply(
        lambda df: pd.Series({
            "n_scenarios": df[["model", "scenario"]].drop_duplicates().shape[0],
            "median_of_medians": df["median"].median(),
            "p34_of_p34": df["p34"].quantile(0.34),
            "p67_of_p67": df["p67"].quantile(0.67),
            "p83_of_p83": df["p83"].quantile(0.83),
            "p5_of_p5": df["p5"].quantile(0.05),
            "p10_of_p10": df["p10"].quantile(0.1),
            "p17_of_p17": df["p17"].quantile(0.17),
            "p95_of_p95": df["p95"].quantile(0.95),
            "p90_of_p90": df["p90"].quantile(0.9),
        })
    ).reset_index()

    return group_summary

## do the stats on all scenarios for NZCO2 year

In [7]:
# Per-scenario statistics
scenario_stats = calculate_scenario_stats(metrics_df, "GSAT|ANTHROPOGENIC|delta_netzero_co2")

# Group summary
scenarios_summary = calculate_group_summary(scenario_stats)

/var/folders/y4/5qztk4vn4fq27tgyj0wzy83m0000gn/T/ipykernel_16896/2154116850.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_summary = scenario_stats.groupby("scenario_group").apply(


In [8]:
scenarios_summary

,scenario_group,n_scenarios,median_of_medians,p34_of_p34,p67_of_p67,p83_of_p83,p5_of_p5,p10_of_p10,p17_of_p17,p95_of_p95,p90_of_p90
0,Scenarios (NZCO2),784.0,-0.111048,-0.193707,-0.026464,0.034803,-0.496874,-0.386838,-0.310966,0.180448,0.087595
1,Scenarios (NZGHG),387.0,-0.192568,-0.268703,-0.119301,-0.023953,-0.540348,-0.452916,-0.372091,0.114550,0.042775
2,Stylised Variants (NZCO2-hold),784.0,-0.047248,-0.103381,0.001711,0.081359,-0.277852,-0.220934,-0.174833,0.273587,0.164744
3,Stylised Variants (NZGHG-hold),387.0,-0.148756,-0.214298,-0.088410,0.006789,-0.428578,-0.346477,-0.287503,0.159808,0.071522


## Combined Summary

In [9]:
groups = ["Scenarios (NZCO2)", "Scenarios (NZGHG)", "Stylised Variants (NZCO2-hold)", "Stylised Variants (NZGHG-hold)"]

def format_cell(row):
    """Format as median with ranges underneath."""
    if row is None:
        return ["—", "", "", "" , ""]
    return [
        f"{row['median_of_medians']:.2f}",
        f"({row['p34_of_p34']:.2f} to {row['p67_of_p67']:.2f})",
        f"({row['p17_of_p17']:.2f} to {row['p83_of_p83']:.2f})",
        f"({row['p10_of_p10']:.2f} to {row['p90_of_p90']:.2f})",
        f"({row['p5_of_p5']:.2f} to {row['p95_of_p95']:.2f})"
    ]

# Index scenarios_summary by scenario_group for easy lookup
group_summaries = scenarios_summary.set_index("scenario_group")

rows = []
for stat_label, val_idx in [
    ("Median of medians", 0),
    ("(p34 to p67)", 1),
    ("(p17 to p83)", 2),
    ("(p10 to p90)", 3),
    ("(p05 to p95)", 4),
]:
    row = {"Statistic": stat_label}
    for grp in groups:
        if grp in group_summaries.index:
            vals = format_cell(group_summaries.loc[grp])
        else:
            vals = ["—", "", "", "", ""]
        row[grp] = vals[val_idx]
    rows.append(row)

summary_table = pd.DataFrame(rows).set_index("Statistic")

# Scenario counts
counts = {
    grp: int(group_summaries.loc[grp, "n_scenarios"]) if grp in group_summaries.index else 0
    for grp in groups
}

print("Delta Warming (NZCO2 year to 2100) [°C]")
print("Scenario counts: " + ", ".join(f"{grp} (n={counts[grp]})" for grp in groups))
print()

Delta Warming (NZCO2 year to 2100) [°C]
Scenario counts: Scenarios (NZCO2) (n=784), Scenarios (NZGHG) (n=387), Stylised Variants (NZCO2-hold) (n=784), Stylised Variants (NZGHG-hold) (n=387)



In [10]:
summary_table.columns

Index(['Scenarios (NZCO2)', 'Scenarios (NZGHG)',
       'Stylised Variants (NZCO2-hold)', 'Stylised Variants (NZGHG-hold)'],
      dtype='object')

In [11]:
rename_map = {
    "Scenarios (NZCO2)":         "Scenarios\n(NZCO2)",
    "Scenarios (NZGHG)":         "Scenarios\n(NZGHG)",
    "Stylised Variants (NZCO2-hold)": "Stylised Variants\n(NZCO2-hold)",
    "Stylised Variants (NZGHG-hold)": "Stylised Variants\n(NZGHG-hold)",
}

summary_table.columns = summary_table.columns.map(rename_map)

In [12]:
fig, ax = plt.subplots(figsize=(6, len(summary_table) * 0.2))
ax.axis("off")

table = ax.table(
    cellText=summary_table.values,
    colLabels=summary_table.columns,
    rowLabels=[f"{col} [°C]" for col in summary_table.index],
    loc="center",
    cellLoc="center",  # centers cell contents
    colLoc="center",   # centers column headers
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.2, 1)

header_height = 0.37  # adjust to taste
for col_idx in range(len(summary_table.columns)):
    table[0, col_idx].set_height(header_height)
    table[0, col_idx].get_text().set_fontweight("bold")
    table[0, col_idx].set_facecolor("#e0e0e0")
    
# Bold column headers (row index 0)
for col_idx in range(len(summary_table.columns)):
    table[0, col_idx].get_text().set_fontweight("bold")
    table[0, col_idx].set_facecolor("#e0e0e0")  # light grey background

fig.savefig(PLOT_FOLDER / "502_decline_stats_nzco2_table.png", bbox_inches="tight", dpi=300)
plt.close(fig)

## do the stats on all scenarios for peak GSAT year

In [13]:
# Per-scenario statistics
scenario_stats_gsat = calculate_scenario_stats(metrics_df, "GSAT|ANTHROPOGENIC|delta_peak_gsat")

# Group summary
scenarios_summary_gsat = calculate_group_summary(scenario_stats_gsat)

/var/folders/y4/5qztk4vn4fq27tgyj0wzy83m0000gn/T/ipykernel_16896/2154116850.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_summary = scenario_stats.groupby("scenario_group").apply(


In [14]:
groups = ["Scenarios (NZCO2)", "Scenarios (NZGHG)", "Stylised Variants (NZCO2-hold)", "Stylised Variants (NZGHG-hold)"]

def format_cell(row):
    """Format as median with ranges underneath."""
    if row is None:
        return ["—", "", "", ""]
    return [
        f"{row['median_of_medians']:.2f}",
        f"({row['p34_of_p34']:.2f} to {row['p67_of_p67']:.2f})",
        f"({row['p17_of_p17']:.2f} to {row['p83_of_p83']:.2f})",
        f"({row['p10_of_p10']:.2f} to {row['p90_of_p90']:.2f})",
        f"({row['p5_of_p5']:.2f} to {row['p95_of_p95']:.2f})"
    ]
    
# Index scenarios_summary by scenario_group for easy lookup
group_summaries_gsat = scenarios_summary_gsat.set_index("scenario_group")

rows = []
for stat_label, val_idx in [
    ("Median of medians", 0),
    ("(p34 to p67)", 1),
    ("(p17 to p83)", 2),
    ("(p10 to p90)", 3),
    ("(p05 to p95)", 4),
]:
    row = {"Statistic": stat_label}
    for grp in groups:
        if grp in group_summaries_gsat.index:
            vals = format_cell(group_summaries_gsat.loc[grp])
        else:
            vals = ["—", "", "", ""]
        row[grp] = vals[val_idx]
    rows.append(row)

summary_table_gsat = pd.DataFrame(rows).set_index("Statistic")

# Scenario counts
counts = {
    grp: int(group_summaries_gsat.loc[grp, "n_scenarios"]) if grp in group_summaries_gsat.index else 0
    for grp in groups
}

print("Delta Warming (NZCO2 year to 2100) [°C]")
print("Scenario counts: " + ", ".join(f"{grp} (n={counts[grp]})" for grp in groups))
print()
summary_table_gsat[groups]

Delta Warming (NZCO2 year to 2100) [°C]
Scenario counts: Scenarios (NZCO2) (n=784), Scenarios (NZGHG) (n=387), Stylised Variants (NZCO2-hold) (n=784), Stylised Variants (NZGHG-hold) (n=387)



,Scenarios (NZCO2),Scenarios (NZGHG),Stylised Variants (NZCO2-hold),Stylised Variants (NZGHG-hold)
Statistic,,,,
Median of medians,-0.12,-0.21,-0.07,-0.17
(p34 to p67),(-0.22 to -0.04),(-0.30 to -0.13),(-0.13 to -0.01),(-0.24 to -0.09)
(p17 to p83),(-0.35 to 0.00),(-0.41 to -0.04),(-0.22 to 0.00),(-0.32 to -0.02)
(p10 to p90),(-0.43 to 0.00),(-0.48 to -0.01),(-0.27 to 0.00),(-0.38 to 0.00)
(p05 to p95),(-0.53 to 0.00),(-0.56 to 0.00),(-0.33 to 0.00),(-0.45 to 0.00)


In [15]:
summary_table_gsat.columns = summary_table_gsat.columns.map(rename_map)


fig, ax = plt.subplots(figsize=(6, len(summary_table_gsat) * 0.2))
ax.axis("off")

table = ax.table(
    cellText=summary_table_gsat.values,
    colLabels=summary_table_gsat.columns,
    rowLabels=[f"{col} [°C]" for col in summary_table_gsat.index],
    loc="center",
    cellLoc="center",  # centers cell contents
    colLoc="center",   # centers column headers
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.2, 1)

header_height = 0.37  # adjust to taste
for col_idx in range(len(summary_table.columns)):
    table[0, col_idx].set_height(header_height)
    table[0, col_idx].get_text().set_fontweight("bold")
    table[0, col_idx].set_facecolor("#e0e0e0")

# Bold column headers (row index 0)
for col_idx in range(len(summary_table_gsat.columns)):
    table[0, col_idx].get_text().set_fontweight("bold")
    table[0, col_idx].set_facecolor("#e0e0e0")  # light grey background
    
fig.savefig(PLOT_FOLDER / "502_decline_stats_gsat_table.png", bbox_inches="tight", dpi=300)
plt.close(fig)